This notebook contains code for analyzing the hardware data relating the clusterstate preparation and its fidelity on IBM hardware. Here we do a measurement-based approach, where we prepare a graph state native to the hardware connectivity and then prepare a cluster state by performing X measurements on the appropriate qubits. We then analyze the resulting data to extract the fidelity of the prepared cluster state. Compare  https://iopscience.iop.org/article/10.1088/1367-2630/ad51e5/meta for more details.

In [1]:
from pauli_measurements import *
from qiskit.providers.fake_provider import FakeCairoV2, FakeSherbrooke

import numpy as np
import pickle

import sympy as sp
theta = sp.Symbol('theta')
eta = sp.Symbol('eta')
Z = sp.Matrix([[1, 0], [0, -1]])
YY = sp.Matrix([[0, -1j], [1j, 0]])
Y = sp.sqrt(1j*YY).as_mutable()
A = sp.sqrt(-1j*YY).as_mutable()
P = sp.sqrt(1j*Z).as_mutable()
M = sp.sqrt(-1j*Z).as_mutable()
H = sp.Matrix([[1, 1], [1, -1]])/sp.sqrt(2)
X = sp.Matrix([[0, 1], [1, 0]])
I = sp.Matrix([[1, 0], [0, 1]])
S = sp.Matrix([[1, 0], [0, 1j]])
Rx = sp.Matrix([[sp.cos(theta), -1j*sp.sin(theta)], [-1j*sp.sin(theta), sp.cos(theta)]])
Phase = sp.Matrix([[1, 0], [0, sp.exp(1j*eta)]])

# generate dict that contains all matrices defined above where keys are the names of the matrices
LOC_MAT = {'Z': np.array(sp.Matrix([[1, 0], [0, -1]]).evalf()).astype(complex),
                'P': np.array(sp.sqrt(1j*Z).evalf()).astype(complex),
                'M': np.array(sp.sqrt(-1j*Z).evalf()).astype(complex),
                'I': np.array(sp.Matrix([[1, 0], [0, 1]])).astype(complex),
                }

# first key is the measured observable, second key is the the effect on the overall sign depending on the 
# additional operator applied to the measured qubit
# note that first ev of Y measurements corresponds to "-" ev.

MEAS_RES_SIGNS = {
                             'X': {'I': [1,-1], 'Z': [-1,1], 'P': [-1,1], 'M': [1,-1]},
                             'Y': {'I': [1,-1], 'Z': [-1,1], 'P': [-1,1], 'M': [1,-1]},
                             'Z': {'I': [1,-1], 'Z': [1,-1], 'P': [1,-1], 'M': [1,-1]},
                             'I': {'I': [1,1], 'Z': [1,1], 'P': [1,1], 'M': [1,1]}
                             }

MEAS_RES_SIGNS_NO_CORR = {
                             'X': {'I': [1,-1], 'Z': [1,-1], 'P': [1,-1], 'M': [1,-1]},
                             'Y': {'I': [1,-1], 'Z': [1,-1], 'P': [1,-1], 'M': [1,-1]},
                             'Z': {'I': [1,-1], 'Z': [1,-1], 'P': [1,-1], 'M': [1,-1]},
                             'I': {'I': [1,1], 'Z': [1,1], 'P': [1,1], 'M': [1,1]}
                             }

def string_to_sign(string):
    return [-1 if i == '1' else 1 for i in string]  

# compare if matrix A and B are equal up to a global arbitrary phase
def compare_matrices(A,B, verbose=False):
    A_nonzero = A[np.abs(A) > 1e-10]
    B_nonzero = B[np.abs(B) > 1e-10]
    if np.alltrue(np.stack(np.nonzero(A), axis=-1) == np.stack(np.nonzero(B), axis=-1)):
        a = np.nan_to_num(A/B)
        a = a[np.invert(np.isclose(a,0))]
        if np.all(np.isclose(a,a[0])):
            return True 
        else:
            if verbose:
                print('no global phase')
    else:
        if verbose:
            print('support of matrices not equal')
    return False


def comp_pauli_meas(test_int, edge_list, X_meas_ind, Y_meas_ind, Z_meas_ind):
    test_sign = string_to_sign(test_int)
    
    test_graph = ClusterState()
    test_graph.initialize_graph_from_edge_list(edge_list)
    # do Z measurements to discard unused qubits from nework. In real hardware, these would not be part of the network in the first place.
    for i in Z_meas_ind:
        Z_measurement(test_graph, i, meas_sign=test_sign[i])  
    for i in X_meas_ind:
        X_measurement(test_graph,i[0],i[1], meas_sign=test_sign[i[0]])
        X_measurement(test_graph,i[1], meas_sign=test_sign[i[1]])
    for i in Y_meas_ind:
        Y_measurement(test_graph,i,meas_sign= test_sign[i])
    return test_graph

def get_effective_deco(test_graph, list_graph_qubits):
    effective_deco = {i: 0 for i in list_graph_qubits}
    for q in list_graph_qubits:
        deco = test_graph.decorations[q]
        mat = np.eye(2)
        for i in deco:
            mat = np.matmul(mat,LOC_MAT[i])
        
        if compare_matrices(mat, np.asanyarray(I).astype(complex)):
            effective_deco[q] = 'I'
        elif compare_matrices(mat, np.asanyarray(Z).astype(complex)):
            effective_deco[q] ='Z'
        elif compare_matrices(mat, np.asanyarray(P).astype(complex)):
            effective_deco[q] = 'P'
        elif compare_matrices(mat, np.asanyarray(M).astype(complex)):
            effective_deco[q] = 'M'
        else:
            effective_deco[q] = test_graph.decorations[q]
    return effective_deco

def add_counts(exp_counts):
    all_counts = {}
    for counts in exp_counts:
        for c in counts:
            if c in all_counts:
                all_counts[c] += counts[c]
            else:
                all_counts.update({c: counts[c]})
    for counts in all_counts:
        all_counts[counts] = all_counts[counts]/len(exp_counts)
    return all_counts

Create date for a scatter plot containing:

2*3 graph state: vanilla, rand dd our method + naive implementation vanilla, rand dd

In [2]:
term11 = [('X',1),('Z',3),('Z',12)]
term12 = [('Y',14),('Z',3),('Z',12),('Z',25)]
term13 = [('X',23),('Z',12),('Z',25)]

stab1 = [term11, term12, term13]


term21 = [('X',3),('Z',1),('Z',14)]
term22 = [('Y',12),('Z',1),('Z',14),('Z',23)]
term23 = [('X',25),('Z',23),('Z',14)]

stab2 = [term21, term22, term23]


In [ ]:
# Load data using pickle

with open('new_ehningen/ehningen_ressource_randomized_dd.pickle', 'rb') as handle:
    qc_data = pickle.load(handle)
qc_data = add_counts(qc_data['counts_qc'][0:100])

all_keys = [i for i in qc_data]

backend = FakeCairoV2()
edge_list = list(backend.coupling_map.get_edges())
X_meas_ind = [[4,7],[15,18],[16,19],[5,8]]
Y_meas_ind = [10, 21, 11, 22, 2, 13, 24]
Z_meas_ind = [0,6,17,9,20,26]
list_graph_qubits = [1,3,12,14,23,25]

expval_per_shot = []
moredata = []
all_effective_decos = []
for test_int in all_keys:

    test_graph = comp_pauli_meas(test_int, edge_list, X_meas_ind, Y_meas_ind, Z_meas_ind)
    effective_deco = get_effective_deco(test_graph, list_graph_qubits)
    all_effective_decos.append(effective_deco)

    sign_per_shot = []
    for term in stab1:
        sign = 1
        for obs, q in term:
            sign *= MEAS_RES_SIGNS[obs][effective_deco[q]][int(test_int[q])]
        sign_per_shot.append(sign)
    moredata.append([s*qc_data[test_int] for s in sign_per_shot])
    expval_per_shot.append(np.sum(sign_per_shot))

exp_val_stab1 = []
for i,key in enumerate(all_keys):
    exp_val_stab1.append(qc_data[key]*expval_per_shot[i])
    
print('expval stab1: ', np.sum(exp_val_stab1))

moredata = np.array(moredata)
mean_per_term1 = np.sum(moredata, axis=0)



with open('new_ehningen/ehningen_ressource_randomized_dd.pickle', 'rb') as handle:
    qc_data = pickle.load(handle)
qc_data = add_counts(qc_data['counts_qc'][100:])

all_keys = [i for i in qc_data]

backend = FakeCairoV2()
edge_list = list(backend.coupling_map.get_edges())
X_meas_ind = [[4,7],[15,18],[16,19],[5,8]]
Y_meas_ind = [10, 21, 11, 22, 2, 13, 24]
Z_meas_ind = [0,6,17,9,20,26]
list_graph_qubits = [1,3,12,14,23,25]

expval_per_shot = []
moredata = []
all_effective_decos = []
for test_int in all_keys:

    test_graph = comp_pauli_meas(test_int, edge_list, X_meas_ind, Y_meas_ind, Z_meas_ind)
    effective_deco = get_effective_deco(test_graph, list_graph_qubits)
    all_effective_decos.append(effective_deco)

    sign_per_shot = []
    for term in stab2:
        sign = 1
        for obs, q in term:
            sign *= MEAS_RES_SIGNS[obs][effective_deco[q]][int(test_int[q])]
        sign_per_shot.append(sign)
    moredata.append([s*qc_data[test_int] for s in sign_per_shot])
    expval_per_shot.append(np.sum(sign_per_shot))

exp_val_stab = []
for i,key in enumerate(all_keys):
    exp_val_stab.append(qc_data[key]*expval_per_shot[i])
    
print('expval stab2: ', np.sum(exp_val_stab))

moredata = np.array(moredata)
mean_per_term2 = np.sum(moredata, axis=0)

np.save('2x3_randomized_dd.npy', np.concatenate((mean_per_term1,mean_per_term2), axis=0))


/var/folders/bt/xt5qzyzs1_d64bk976sgjk0w0000gn/T/ipykernel_93702/1087693287.py:56: RuntimeWarning: invalid value encountered in divide
  a = np.nan_to_num(A/B)


expval stab1:  1.7789998784579364
expval stab2:  1.7160061128128548


In [ ]:
# Load data using pickle
with open('new_ehningen/ehningen_ressource.pickle', 'rb') as handle:
    qc_data = pickle.load(handle)
qc_data = qc_data['counts_qc'][0]

all_keys = [i for i in qc_data]

backend = FakeCairoV2()
edge_list = list(backend.coupling_map.get_edges())
X_meas_ind = [[4,7],[15,18],[16,19],[5,8]]
Y_meas_ind = [10, 21, 11, 22, 2, 13, 24]
Z_meas_ind = [0,6,17,9,20,26]
list_graph_qubits = [1,3,12,14,23,25]

expval_per_shot = []
moredata = []
all_effective_decos = []
for test_int in all_keys:

    test_graph = comp_pauli_meas(test_int, edge_list, X_meas_ind, Y_meas_ind, Z_meas_ind)
    effective_deco = get_effective_deco(test_graph, list_graph_qubits)
    all_effective_decos.append(effective_deco)

    sign_per_shot = []
    for term in stab1:
        sign = 1
        for obs, q in term:
            sign *= MEAS_RES_SIGNS[obs][effective_deco[q]][int(test_int[q])]
        sign_per_shot.append(sign)
    moredata.append([s*qc_data[test_int] for s in sign_per_shot])
    expval_per_shot.append(np.sum(sign_per_shot))

exp_val_stab1 = []
for i,key in enumerate(all_keys):
    exp_val_stab1.append(qc_data[key]*expval_per_shot[i])
    
print('expval stab1: ', np.sum(exp_val_stab1))

moredata = np.array(moredata)
mean_per_term1 = np.sum(moredata, axis=0)


with open('new_ehningen/ehningen_ressource.pickle', 'rb') as handle:
    qc_data = pickle.load(handle)
qc_data = qc_data['counts_qc'][1]

# with open('ehingen_resource.pickle', 'rb') as handle:
#     qc_data = pickle.load(handle)
# qc_data = qc_data[1]

all_keys = [i for i in qc_data]

backend = FakeCairoV2()
edge_list = list(backend.coupling_map.get_edges())
X_meas_ind = [[4,7],[15,18],[16,19],[5,8]]
Y_meas_ind = [10, 21, 11, 22, 2, 13, 24]
Z_meas_ind = [0,6,17,9,20,26]
list_graph_qubits = [1,3,12,14,23,25]

expval_per_shot = []
moredata = []
all_effective_decos = []
for test_int in all_keys:

    test_graph = comp_pauli_meas(test_int, edge_list, X_meas_ind, Y_meas_ind, Z_meas_ind)
    effective_deco = get_effective_deco(test_graph, list_graph_qubits)
    all_effective_decos.append(effective_deco)

    sign_per_shot = []
    for term in stab2:
        sign = 1
        for obs, q in term:
            sign *= MEAS_RES_SIGNS[obs][effective_deco[q]][int(test_int[q])]
        sign_per_shot.append(sign)
    moredata.append([s*qc_data[test_int] for s in sign_per_shot])
    expval_per_shot.append(np.sum(sign_per_shot))

exp_val_stab = []
for i,key in enumerate(all_keys):
    exp_val_stab.append(qc_data[key]*expval_per_shot[i])
    
print('expval stab: ', np.sum(exp_val_stab))

moredata = np.array(moredata)
mean_per_term2 = np.sum(moredata, axis=0)

np.save('2x3.npy', np.concatenate((mean_per_term1,mean_per_term2), axis=0))

/var/folders/bt/xt5qzyzs1_d64bk976sgjk0w0000gn/T/ipykernel_84086/4157475335.py:59: RuntimeWarning: invalid value encountered in divide
  a = np.nan_to_num(A/B)


expval stab1:  1.702634811082914
expval stab:  1.7567205997787791


In [18]:
# stabilizer for 127 exp

stab_127_1 = [
    [('X',0),('Z',4),('Z',20)],
    [('Y',8),('Z',4),('Z',26),('Z',12)],
    [('X',22),('Z',4),('Z',26),('Z',20),('Z',43)],
    [('Y',30),('Z',12),('Z',26),('Z',49)],
    [('Y',39),('Z',20),('Z',43),('Z',58)],
    [('X',47),('Z',26),('Z',43),('Z',49),('Z',64)],
    [('X',60),('Z',58),('Z',43),('Z',81),('Z',64)],
    [('Y',68),('Z',49),('Z',64),('Z',87)],
    [('Y',77),('Z',58),('Z',81),('Z',96)],
    [('X',85),('Z',87),('Z',104),('Z',81),('Z',64)],
    [('X',100),('Z',96),('Z',104),('Z',81),('Z',118)],
    [('Y',106),('Z',87),('Z',104),('Z',124)],
    [('X',114),('Z',96),('Z',118)],
    [('Y',122),('Z',118),('Z',104),('Z',124)]
]

stab_127_2 = [
    [('Y',4),('Z',0),('Z',8),('Z',22)],
    [('X',12),('Z',8),('Z',30)],
    [('Y',20),('Z',22),('Z',0),('Z',39)],
    [('X',26),('Z',8),('Z',30),('Z',22),('Z',47)],
    [('X',43),('Z',22),('Z',39),('Z',60),('Z',47)],
    [('Y',49),('Z',47),('Z',30),('Z',68)],
    [('Y',58),('Z',39),('Z',60),('Z',77)],
    [('X',64),('Z',68),('Z',47),('Z',85),('Z',60)],
    [('X',81),('Z',77),('Z',85),('Z',60),('Z',100)],
    [('Y',87),('Z',68),('Z',85),('Z',106)],
    [('Y',96),('Z',77),('Z',100),('Z',114)],
    [('X',104),('Z',106),('Z',122),('Z',85),('Z',100)],
    [('Y',118),('Z',114),('Z',122),('Z',100)],
    [('X',124),('Z',122),('Z',106)]
]

In [ ]:
# analyze new sherbrook data (14.05.2024)
# Load data using pickle
# with open('sherbrooke/results_sherbrooke.pickle', 'rb') as f:
with open('sherbrooke/results_sherbrooke_randomized_dd.pickle', 'rb') as f:
    qc_data = pickle.load(f)
# qc_data = qc_data['counts_qc'][0]
qc_data = add_counts(qc_data['counts_qc'][0:100])
all_keys = [i for i in qc_data]
# all_keys = [i for i in qc_data[0]]

backend = FakeSherbrooke()
Sherbrooke_coupling_map = backend.coupling_map
edge_list = list(Sherbrooke_coupling_map.get_edges())

Z_meas_ind = [13,113]
X_meas_ind = [[1,2],[5,6],[9,10],[14,18],[24,23],[28,27],[31,32],[36,51],
              [45,46],[38,37],[52,56],[42,41],[69,70],[74,89],[76,75],
              [90,94],[107,108],[112,126],[115,116],[119,120],[61,62],
              [65,66],[84,83],[80,79],[103,102],[99,98]  
              ]
Y_meas_ind = [3,7,11,19,15,16,17,21,25,29,33,34,35,50,40,48,
              44,57,53,55,88,71,95,93,125,109,117,121,123,110,
              111,59,54,63,67,72,73,86,82,78,91,92,105,101,97
              ]
list_graph_qubits = [0,4,8,12,20,22,26,30,39,43,47,49,58,60,64,68,77,81,
                     85,87,96,100,104,106,114,118,122,124]

expval_per_shot = []
moredata = []
all_effective_decos = []
for test_int in all_keys:

    test_graph = comp_pauli_meas(test_int, edge_list, X_meas_ind, Y_meas_ind, Z_meas_ind)
    effective_deco = get_effective_deco(test_graph, list_graph_qubits)
    all_effective_decos.append(effective_deco)

    sign_per_shot = []
    for term in stab_127_1:
        sign = 1
        for obs, q in term:
            sign *= MEAS_RES_SIGNS[obs][effective_deco[q]][int(test_int[q])]
        sign_per_shot.append(sign)
    moredata.append([s*qc_data[test_int] for s in sign_per_shot])
    expval_per_shot.append(np.sum(sign_per_shot))

exp_val_stab1 = []
for i,key in enumerate(all_keys):
    exp_val_stab1.append(qc_data[key]*expval_per_shot[i])
    
print('expval stab1: ', np.sum(exp_val_stab1))

moredata = np.array(moredata)
mean_per_term1 = np.sum(moredata, axis=0)

with open('sherbrooke/results_sherbrooke_randomized_dd.pickle', 'rb') as f:
# with open('sherbrooke/results_sherbrooke.pickle', 'rb') as f:
    qc_data = pickle.load(f)
# qc_data = qc_data['counts_qc'][1]
qc_data = add_counts(qc_data['counts_qc'][100:])

all_keys = [i for i in qc_data]
# all_keys = [i for i in qc_data[1]]

backend = FakeSherbrooke()
Sherbrooke_coupling_map = backend.coupling_map
edge_list = list(Sherbrooke_coupling_map.get_edges())

Z_meas_ind = [13,113]
X_meas_ind = [[1,2],[5,6],[9,10],[14,18],[24,23],[28,27],[31,32],[36,51],
              [45,46],[38,37],[52,56],[42,41],[69,70],[74,89],[76,75],
              [90,94],[107,108],[112,126],[115,116],[119,120],[61,62],
              [65,66],[84,83],[80,79],[103,102],[99,98]  
              ]
Y_meas_ind = [3,7,11,19,15,16,17,21,25,29,33,34,35,50,40,48,
              44,57,53,55,88,71,95,93,125,109,117,121,123,110,
              111,59,54,63,67,72,73,86,82,78,91,92,105,101,97
              ]
list_graph_qubits = [0,4,8,12,20,22,26,30,39,43,47,49,58,60,64,68,77,81,
                     85,87,96,100,104,106,114,118,122,124]

expval_per_shot = []
moredata = []
all_effective_decos = []
for test_int in all_keys:

    test_graph = comp_pauli_meas(test_int, edge_list, X_meas_ind, Y_meas_ind, Z_meas_ind)
    effective_deco = get_effective_deco(test_graph, list_graph_qubits)
    all_effective_decos.append(effective_deco)

    sign_per_shot = []
    for term in stab_127_2:
        sign = 1
        for obs, q in term:
            sign *= MEAS_RES_SIGNS[obs][effective_deco[q]][int(test_int[q])]
        sign_per_shot.append(sign)
    moredata.append([s*qc_data[test_int] for s in sign_per_shot])
    expval_per_shot.append(np.sum(sign_per_shot))

exp_val_stab = []
for i,key in enumerate(all_keys):
    exp_val_stab.append(qc_data[key]*expval_per_shot[i])
    
print('expval stab: ', np.sum(exp_val_stab))

moredata = np.array(moredata)
mean_per_term2 = np.sum(moredata, axis=0)

np.save('4x7_sherbrooke_randomized_dd.npy', np.concatenate((mean_per_term1,mean_per_term2), axis=0))

/var/folders/bt/xt5qzyzs1_d64bk976sgjk0w0000gn/T/ipykernel_84086/4157475335.py:59: RuntimeWarning: invalid value encountered in divide
  a = np.nan_to_num(A/B)


expval stab1:  5.366
expval stab:  6.754200000000001


In [ ]:
# analyze new sherbrook data (14.05.2024)
# Load data using pickle
with open('sherbrooke/results_sherbrooke.pickle', 'rb') as f:
    qc_data = pickle.load(f)
qc_data = qc_data['counts_qc'][0]
# qc_data = add_counts(qc_data['counts_qc'][0:100])
all_keys = [i for i in qc_data]
# all_keys = [i for i in qc_data[0]]

backend = FakeSherbrooke()
Sherbrooke_coupling_map = backend.coupling_map
edge_list = list(Sherbrooke_coupling_map.get_edges())

Z_meas_ind = [13,113]
X_meas_ind = [[1,2],[5,6],[9,10],[14,18],[24,23],[28,27],[31,32],[36,51],
              [45,46],[38,37],[52,56],[42,41],[69,70],[74,89],[76,75],
              [90,94],[107,108],[112,126],[115,116],[119,120],[61,62],
              [65,66],[84,83],[80,79],[103,102],[99,98]  
              ]
Y_meas_ind = [3,7,11,19,15,16,17,21,25,29,33,34,35,50,40,48,
              44,57,53,55,88,71,95,93,125,109,117,121,123,110,
              111,59,54,63,67,72,73,86,82,78,91,92,105,101,97
              ]
list_graph_qubits = [0,4,8,12,20,22,26,30,39,43,47,49,58,60,64,68,77,81,
                     85,87,96,100,104,106,114,118,122,124]

expval_per_shot = []
moredata = []
all_effective_decos = []
for test_int in all_keys:

    test_graph = comp_pauli_meas(test_int, edge_list, X_meas_ind, Y_meas_ind, Z_meas_ind)
    effective_deco = get_effective_deco(test_graph, list_graph_qubits)
    all_effective_decos.append(effective_deco)

    sign_per_shot = []
    for term in stab_127_1:
        sign = 1
        for obs, q in term:
            sign *= MEAS_RES_SIGNS[obs][effective_deco[q]][int(test_int[q])]
        sign_per_shot.append(sign)
    moredata.append([s*qc_data[test_int] for s in sign_per_shot])
    expval_per_shot.append(np.sum(sign_per_shot))

exp_val_stab1 = []
for i,key in enumerate(all_keys):
    exp_val_stab1.append(qc_data[key]*expval_per_shot[i])
    
print('expval stab1: ', np.sum(exp_val_stab1))

moredata = np.array(moredata)
mean_per_term1 = np.sum(moredata, axis=0)

# with open('sherbrooke/results_sherbrooke_randomized_dd.pickle', 'rb') as f:
with open('sherbrooke/results_sherbrooke.pickle', 'rb') as f:
    qc_data = pickle.load(f)
qc_data = qc_data['counts_qc'][1]
# qc_data = add_counts(qc_data['counts_qc'][100:])

all_keys = [i for i in qc_data]
# all_keys = [i for i in qc_data[1]]

backend = FakeSherbrooke()
Sherbrooke_coupling_map = backend.coupling_map
edge_list = list(Sherbrooke_coupling_map.get_edges())

Z_meas_ind = [13,113]
X_meas_ind = [[1,2],[5,6],[9,10],[14,18],[24,23],[28,27],[31,32],[36,51],
              [45,46],[38,37],[52,56],[42,41],[69,70],[74,89],[76,75],
              [90,94],[107,108],[112,126],[115,116],[119,120],[61,62],
              [65,66],[84,83],[80,79],[103,102],[99,98]  
              ]
Y_meas_ind = [3,7,11,19,15,16,17,21,25,29,33,34,35,50,40,48,
              44,57,53,55,88,71,95,93,125,109,117,121,123,110,
              111,59,54,63,67,72,73,86,82,78,91,92,105,101,97
              ]
list_graph_qubits = [0,4,8,12,20,22,26,30,39,43,47,49,58,60,64,68,77,81,
                     85,87,96,100,104,106,114,118,122,124]

expval_per_shot = []
moredata = []
all_effective_decos = []
for test_int in all_keys:

    test_graph = comp_pauli_meas(test_int, edge_list, X_meas_ind, Y_meas_ind, Z_meas_ind)
    effective_deco = get_effective_deco(test_graph, list_graph_qubits)
    all_effective_decos.append(effective_deco)

    sign_per_shot = []
    for term in stab_127_2:
        sign = 1
        for obs, q in term:
            sign *= MEAS_RES_SIGNS[obs][effective_deco[q]][int(test_int[q])]
        sign_per_shot.append(sign)
    moredata.append([s*qc_data[test_int] for s in sign_per_shot])
    expval_per_shot.append(np.sum(sign_per_shot))

exp_val_stab = []
for i,key in enumerate(all_keys):
    exp_val_stab.append(qc_data[key]*expval_per_shot[i])
    
print('expval stab: ', np.sum(exp_val_stab))

moredata = np.array(moredata)
mean_per_term2 = np.sum(moredata, axis=0)

np.save('4x7_sherbrooke.npy', np.concatenate((mean_per_term1,mean_per_term2), axis=0))

/var/folders/bt/xt5qzyzs1_d64bk976sgjk0w0000gn/T/ipykernel_84086/4157475335.py:59: RuntimeWarning: invalid value encountered in divide
  a = np.nan_to_num(A/B)


expval stab1:  4.059999999999996
expval stab:  5.017599999999996
